In [6]:
import gzip, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
np.random.seed(SEED)
os.makedirs("figures", exist_ok=True)

# Colours: sentiment is a polarity, so red = negative, grey = neutral, blue = positive
NEG, NEU, POS = "#e34948", "#898781", "#2a78d6"
SENT_COLORS = {"negative": NEG, "neutral": NEU, "positive": POS}
BLUE = POS

plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
import pandas as pd
base = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
reviews = pd.read_json(f"{base}/review_categories/All_Beauty.jsonl.gz", lines=True)
meta    = pd.read_json(f"{base}/meta_categories/meta_All_Beauty.jsonl.gz", lines=True)
df = reviews.merge(meta[["parent_asin","title","store","price","average_rating"]],
                   on="parent_asin", how="left", suffixes=("_review","_product"))
print(df.shape); df.head()

(701528, 14)


,rating,title_review,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_product,store,price,average_rating
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,Herbivore - Natural Sea Mist Texturizing Salt ...,HERBIVORE,NaN,4.3
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,All Natural Vegan Dry Shampoo Powder - Eco Fri...,Two Goats Apothecary,NaN,4.0
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,New Road Beauty - Creamsicle - Variety 3 Pack ...,New Road Beauty,21.98,4.4
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,muaowig Ombre Body Wave Bundles 1B Grey Human ...,muaowig,NaN,1.0
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True,Yinhua Electric Nail Drill Kit Portable Profes...,Yinhua,NaN,3.5


In [7]:
CATEGORY = "All_Beauty"
BASE = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEW_FILE = f"{CATEGORY}.jsonl.gz"
META_FILE = f"meta_{CATEGORY}.jsonl.gz"

!wget -q -nc {BASE}/review_categories/{REVIEW_FILE}
!wget -q -nc {BASE}/meta_categories/{META_FILE}
!ls -lh *.jsonl.gz

-rw-r--r-- 1 root root 91M Jan 16  2025 All_Beauty.jsonl.gz
-rw-r--r-- 1 root root 39M Jan 16  2025 meta_All_Beauty.jsonl.gz


In [8]:
def read_jsonl(path, cols):
    """Stream a .jsonl / .jsonl.gz file and keep only `cols` (saves RAM on Colab)."""
    opener = gzip.open if path.endswith(".gz") else open
    rows = []
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            row = {c: r.get(c) for c in cols}
            if "images" in cols:
                row["images"] = len(r.get("images") or [])   # keep only the count
            rows.append(row)
    return pd.DataFrame(rows)

review_cols = ["rating", "title", "text", "images", "asin", "parent_asin",
               "user_id", "timestamp", "helpful_vote", "verified_purchase"]
meta_cols = ["parent_asin", "main_category", "title", "store", "price",
             "average_rating", "rating_number"]

reviews = read_jsonl(REVIEW_FILE, review_cols)
meta = read_jsonl(META_FILE, meta_cols)
print("reviews:", reviews.shape, "| products (meta):", meta.shape)
reviews.head(3)

reviews: (701528, 10) | products (meta): (112590, 7)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",0,B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",0,B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
